In [3]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

In [2]:
class CrashState(TypedDict):
    input : str
    step1 : str
    step2 : str 
    step3 : str

    

In [4]:
def step1(state:CrashState):
    print("Step 1 Executed")
    return {'step1' : 'done','input' : state['input']}

def step2(state:CrashState):
    print("Hanging .... Interrupt the Keyboard to simulate crash")
    time.sleep(30)
    return {'step2' : 'done'}

def step3(state:CrashState):
    print("Step 3 Executed")
    return {'step3' : 'done'}

In [6]:
builder = StateGraph(CrashState)

builder.add_node('step1',step1)
builder.add_node('step2',step2)
builder.add_node('step3',step3)

builder.add_edge(START,'step1')
builder.add_edge('step1','step2')
builder.add_edge('step2','step3')
builder.add_edge('step3',END)


checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [7]:
#Displaying the graph
print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | step1 |    
  +-------+    
      *        
      *        
      *        
  +-------+    
  | step2 |    
  +-------+    
      *        
      *        
      *        
  +-------+    
  | step3 |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [8]:
try :
    print("Running the graph. Interrupt the Keyboard to simulate crash")
    graph.invoke({'input' : 'start'}, config = {'configurable': {'thread_id':1}})
except KeyboardInterrupt:
    print("Graph Interrupted. Now we will time travel to the last checkpoint and resume execution")
    

Running the graph. Interrupt the Keyboard to simulate crash
Step 1 Executed
Hanging .... Interrupt the Keyboard to simulate crash
Graph Interrupted. Now we will time travel to the last checkpoint and resume execution


In [9]:
graph.get_state({'configurable': {'thread_id':1}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f109a07-7d5e-6ddf-8001-146d7ce32a74'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-02-14T12:27:03.512194+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f109a07-7d5a-628b-8000-29e7223b2fd5'}}, tasks=(PregelTask(id='4561760b-ea86-c1bf-1d10-482ef73934de', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [11]:
list(graph.get_state_history({'configurable': {'thread_id':1}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f109a07-7d5e-6ddf-8001-146d7ce32a74'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-02-14T12:27:03.512194+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f109a07-7d5a-628b-8000-29e7223b2fd5'}}, tasks=(PregelTask(id='4561760b-ea86-c1bf-1d10-482ef73934de', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f109a07-7d5a-628b-8000-29e7223b2fd5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-02-14T12:27:03.510264+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f109a07-7d55-689

In [12]:
### RERUN TO SHOW FAULT TOLERANCE RESUME
final_state = graph.invoke(None, config = {'configurable': {'thread_id':1}})

print("Final_state : ",final_state)

Hanging .... Interrupt the Keyboard to simulate crash
Step 3 Executed
Final_state :  {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}
